# Mmvlm4SCD — full pipeline (Parts 1–6)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/koneke55/Mmvlm4SCD/blob/main/notebooks/00-mmvlm4scd-full-pipeline.ipynb)

This notebook concatenates the tutorial stack end-to-end:

| Part | Source notebook | Content |
|------|-----------------|--------|
| 1 | `01-eda-clinical.ipynb` | Synthetic clinical EDA + preprocessor |
| 2 | `02-eda-genomic.ipynb` | Synthetic genomic / genotype MAF tilt |
| 3 | `03-eda-imaging.ipynb` | Synthetic imaging embeddings |
| 4 | `04-multimodal-fusion.ipynb` | `MultimodalSCDModel` + losses |
| 5 | `05_colab_nigeria_ndhs2018.ipynb` | Real Nigeria NDHS 2018 cohort + training |
| 6 | `06_colab_mali_dhs2018.ipynb` | Real Mali DHS 2018 cohort + training |

**Also available separately:** `05_colab_west_africa_experiment.ipynb` follows the **same Nigeria NDHS 2018 code path** as Part 5 (only title/badge differ)—run one or the other on Colab, not both, unless you want duplicate training.

**Google Colab / GPU:** see [Unsloth's Colab guide](https://docs.unsloth.ai/get-started/install/google-colab) ([unsloth.ai](https://unsloth.ai)). This package does **not** install `unsloth`; use the guide for Runtime / T4 tips.

Run cells **top to bottom**. One shared environment setup applies to all parts below.


## 1. Environment setup (Colab or local)

- **Colab:** optional `MMVLM_REPO_URL` for your fork; defaults to upstream.
- Installs this package editable (`pip install -e .`).


In [ ]:
import os
import subprocess
import sys


def _in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


def _find_repo_root(start: str) -> str:
    cur = os.path.abspath(start)
    for _ in range(8):
        if os.path.isdir(os.path.join(cur, "src", "mmvlm4scd")):
            return cur
        parent = os.path.dirname(cur)
        if parent == cur:
            break
        cur = parent
    raise RuntimeError(
        "Could not find mmvlm4scd package root (missing src/mmvlm4scd). "
        "Open the notebook from the repo or run the Colab clone cell."
    )


if _in_colab():
    REPO_URL = os.environ.get(
        "MMVLM_REPO_URL",
        "https://github.com/koneke55/Mmvlm4SCD.git",
    )
    DEST = "/content/Mmvlm4SCD"
    if not os.path.isdir(os.path.join(DEST, "src", "mmvlm4scd")):
        subprocess.check_call(
            ["git", "clone", "--depth", "1", REPO_URL, DEST],
            stdout=subprocess.DEVNULL,
        )
    os.chdir(DEST)
    ROOT = DEST
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
else:
    ROOT = _find_repo_root(os.getcwd())
    os.chdir(ROOT)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])

sys.path.insert(0, os.path.join(ROOT, "src"))
print("Repo root:", ROOT)


## 2. Accelerator check

Mirrors the GPU verification pattern recommended alongside [Unsloth's Colab instructions](https://docs.unsloth.ai/get-started/install/google-colab).


In [ ]:
import torch

print("torch:", torch.__version__)
print("cuda_available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
else:
    print("CPU-only runtime — for GPU follow Unsloth's Colab guide (Runtime → Change runtime type).")


## 3. Imports


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from mmvlm4scd.data import StandardPreprocessor, generate_synthetic_cohort
from mmvlm4scd.data.synthetic import SCDSyntheticConfig


---

## Part 1 — Clinical EDA (synthetic)

Continues from imports above (`matplotlib`, `numpy`, `pandas`, synthetic loaders).


## Load cohort and preprocess


In [ ]:
cfg = SCDSyntheticConfig(n_patients=1200, seed=42)
cohort = generate_synthetic_cohort(cfg)
clin: pd.DataFrame = cohort["clinical"]
clin.head()


In [ ]:
pre = StandardPreprocessor().fit(clin)
x_clin = pre.transform(clin)
print("Preprocessor output_dim (matches MultimodalSCDModel clinical_input_dim):", pre.output_dim)
print("Feature matrix shape:", x_clin.shape)


## Summary statistics


In [ ]:
clin.describe(include="all").T


## Genotype mix (Hb phenotypes)


In [ ]:
vc = clin["genotype"].value_counts(normalize=True).sort_index()
display(vc)
vc.plot(kind="bar", title="Genotype prevalence (synthetic)", rot=45)
plt.ylabel("fraction")
plt.tight_layout()
plt.show()


## Labs vs severity label


In [ ]:
sev = cohort["severity"]
fig, axes = plt.subplots(1, 3, figsize=(11, 3))
for ax, col in zip(axes, ["hb_g_dl", "ldh_u_l", "voc_rate_per_year"]):
    for k in range(3):
        mask = sev == k
        ax.hist(clin.loc[mask, col].values, bins=20, alpha=0.45, label=f"sev {k}")
    ax.set_title(col)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()


---

## Part 2 — Genomic EDA (synthetic)

Adding nothing extra—`generate_synthetic_cohort` is already imported.


In [ ]:
def genotype_props(cfg):
    c = generate_synthetic_cohort(cfg)
    g = c["clinical"]["genotype"].values
    labs, cnt = np.unique(g, return_counts=True)
    return dict(zip(labs, cnt / cnt.sum()))

base = genotype_props(SCDSyntheticConfig(n_patients=8000, seed=0))
wa = genotype_props(SCDSyntheticConfig(n_patients=8000, seed=0, west_africa_rs334_maf=0.13))

labels = sorted(set(base) | set(wa))
x = np.arange(len(labels))
w = 0.35
plt.bar(x - w / 2, [base[k] for k in labels], width=w, label="baseline")
plt.bar(x + w / 2, [wa[k] for k in labels], width=w, label="west_africa_rs334_maf=0.13")
plt.xticks(x, labels, rotation=30, ha="right")
plt.ylabel("fraction")
plt.title("Synthetic genotype mix: baseline vs West Africa MAF tilt")
plt.legend()
plt.tight_layout()
plt.show()


## Variant-indicator block (first 16 dims)

Column 0 is aligned with higher genotype severity in the simulator.


In [ ]:
cohort = generate_synthetic_cohort(SCDSyntheticConfig(n_patients=2000, seed=1))
g = cohort["genomic"]
print("genomic shape (N, 32):", g.shape)
print("mean allele-indicator dims 0–15:", g[:, :16].mean(axis=0)[:8])


---

## Part 3 — Imaging EDA (synthetic)


In [ ]:
cohort = generate_synthetic_cohort(SCDSyntheticConfig(n_patients=1500, seed=3))
img = cohort["imaging"]
sev = cohort["severity"]

plt.figure(figsize=(6, 4))
for k in range(3):
    m = sev == k
    plt.hist(img[m, 0], bins=30, alpha=0.45, density=True, label=f"severity {k}")
plt.xlabel("imaging[:, 0] (embedding dim)")
plt.ylabel("density")
plt.title("First embedding dimension vs severity")
plt.legend()
plt.tight_layout()
plt.show()

r = np.corrcoef(img[:, 0], sev)[0, 1]
print(f"Pearson corr(imaging[:,0], severity): {r:.3f}")


## Imaging dimension vs LDH (simulator couples dim 1 to labs)


In [ ]:
ldh = cohort["clinical"]["ldh_u_l"].values
plt.scatter(ldh, img[:, 1], c=sev, alpha=0.35, cmap="viridis")
plt.colorbar(label="severity")
plt.xlabel("LDH")
plt.ylabel("imaging[:, 1]")
plt.title("Embedding dim 1 vs LDH (colored by severity)")
plt.tight_layout()
plt.show()


---

## Part 4 — Multimodal fusion + training losses


**Imports:** Part 4 assumes `torch` and `mmvlm4scd.models` are available. You already imported `torch` in the accelerator cell; run the block below if you restarted the runtime.


In [ ]:
import torch
from torch.utils.data import DataLoader

from mmvlm4scd.data import (
    MultimodalSCDDataset,
    StandardPreprocessor,
    generate_synthetic_cohort,
)
from mmvlm4scd.data.synthetic import SCDSyntheticConfig
from mmvlm4scd.models import ModelConfig, MultimodalSCDModel


In [ ]:
cohort = generate_synthetic_cohort(SCDSyntheticConfig(n_patients=128, seed=0))
pre = StandardPreprocessor().fit(cohort["clinical"])
x_clin = pre.transform(cohort["clinical"])

ds = MultimodalSCDDataset(
    clinical=x_clin,
    genomic=cohort["genomic"],
    imaging=cohort["imaging"],
    temporal=cohort["temporal"],
    severity=cohort["severity"],
    survival_time=cohort["survival_time"],
    survival_event=cohort["survival_event"],
)
loader = DataLoader(ds, batch_size=32, shuffle=False, drop_last=False)
batch = next(iter(loader))
{k: v.shape for k, v in batch.items()}


In [ ]:
def build_model(fusion: str):
    cfg = ModelConfig(
        clinical_input_dim=x_clin.shape[1],
        genomic_input_dim=cohort["genomic"].shape[1],
        imaging_input_dim=cohort["imaging"].shape[1],
        temporal_input_dim=cohort["temporal"].shape[2],
        embed_dim=64,
        fusion=fusion,
    )
    return MultimodalSCDModel(cfg)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

for fusion in ("attention", "cross", "late"):
    model = build_model(fusion).to(device)
    b = {k: v.to(device) for k, v in batch.items()}
    out = model(b)
    print(fusion, "| embedding", tuple(out["embedding"].shape),
          "| severity_logits", tuple(out["severity_logits"].shape),
          "| risk_score", tuple(out["risk_score"].shape))


## Optional: one optimizer step (same stack as unit tests)

Demonstrates that gradients flow through all fusion modes.


In [ ]:
import torch.nn.functional as F

from mmvlm4scd.training.losses import cox_partial_likelihood_loss


def one_step(fusion: str):
    torch.manual_seed(0)
    model = build_model(fusion).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
    b = {k: v.to(device) for k, v in batch.items()}
    opt.zero_grad(set_to_none=True)
    out = model(b)
    loss_cls = F.cross_entropy(out["severity_logits"], b["severity"])
    loss_cox = cox_partial_likelihood_loss(
        out["risk_score"], b["survival_time"], b["survival_event"]
    )
    (loss_cls + 0.1 * loss_cox).backward()
    opt.step()
    return float(loss_cls.detach()), float(loss_cox.detach())

for fusion in ("attention", "cross", "late"):
    lc, lx = one_step(fusion)
    print(f"{fusion}: CE={lc:.4f} Cox={lx:.4f}")


---

## Part 5 — Real cohort: Nigeria NDHS 2018

Requires **NGHR7BDT.dta** (see original notebook prose in `05_colab_nigeria_ndhs2018.ipynb`). Setup cells above already cloned the repo and installed the package.


## 2. Load **real** Nigeria NDHS 2018 rows (Household Recode)

Resolves ``NIGERIA_DHS_NGHR_DTA`` from the environment, or **Colab file upload**. If neither is provided, raises with instructions (no synthetic fallback).

Optional: enforce ``PIP_INDEX_URL`` / offline installs if your runtime blocks PyPI — this section only uses ``pandas``.

In [ ]:
import os
from pathlib import Path

from mmvlm4scd.data import build_cohort_from_nigeria_dhs2018_hr


def resolve_nigeria_dhs_stata_path() -> Path:
    env = os.environ.get("NIGERIA_DHS_NGHR_DTA")
    if env:
        p = Path(env).expanduser()
        if p.is_file():
            return p
        raise FileNotFoundError(f"NIGERIA_DHS_NGHR_DTA points to missing path: {p}")
    cwd = Path("NGHR7BDT.dta").resolve()
    if cwd.is_file():
        return cwd
    if _in_colab():
        from google.colab import files as colab_files  # noqa: WPS433

        print("Upload NGHR7BDT.dta from NGHR7BDT.zip (DHS Nigeria 2018 Household Recode)…")
        up = colab_files.upload()
        names = list(up.keys())
        if len(names) != 1:
            raise RuntimeError("Please upload exactly one .dta file")
        wrote = Path("/content") / names[0]
        wrote.write_bytes(up[names[0]])
        return wrote
    raise FileNotFoundError(
        "Set NIGERIA_DHS_NGHR_DTA to NGHR7BDT.dta or copy that file into the repo root."
    )


TIMESTEPS = 24

dta_path = resolve_nigeria_dhs_stata_path()
print("Stata:", dta_path)

cohort = build_cohort_from_nigeria_dhs2018_hr(
    dta_path,
    timesteps=TIMESTEPS,
    max_patients=None,
)

print(cohort["meta"])
print("Patients:", len(cohort["clinical"]))
cohort["clinical"].head()




## 3. Train and evaluate on real NDHS modalities

Mirrors ``run_full_experiment.py`` but **does not synthesize subjects**. Imaging and temporal modalities are zeros; **survival placeholders** are meaningless here — **`beta=0`** turns off Cox loss (severity cross-entropy only). Tune ``epochs`` / ``batch_size`` for Colab GPU/CPU limits.

In [ ]:
import os
from pathlib import Path

import numpy as np
import torch

from mmvlm4scd.data import StandardPreprocessor
from mmvlm4scd.data.dataloaders import make_loaders
from mmvlm4scd.data.synthetic import split_indices
from mmvlm4scd.evaluation import evaluate_model_full
from mmvlm4scd.models import ModelConfig, MultimodalSCDModel
from mmvlm4scd.training import Trainer, TrainConfig
from mmvlm4scd.utils import auto_device, set_seed


_cfg_seed = int(os.environ.get("MMVLM_SEED", "7"))

cfg = {
    "model": {
        "embed_dim": 64,
        "fusion": "attention",
        "dropout": 0.1,
        "num_severity_classes": 3,
    },
    "train": {
        "epochs": 12,
        "batch_size": 64,
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "grad_clip": 1.0,
        "alpha": 1.0,
        # NDHS cohort: no real survival outcome here — Cox term disabled.
        "beta": 0.0,
        "early_stop_patience": 6,
        "select_metric": "auroc_ovr",
        "device": "cuda" if torch.cuda.is_available() else "cpu",
        "seed": _cfg_seed,
    },
}

set_seed(cfg["train"]["seed"])

pre = StandardPreprocessor().fit(cohort["clinical"])
clin_x = pre.transform(cohort["clinical"])
tr_idx, va_idx, te_idx = split_indices(
    len(cohort["severity"]), seed=_cfg_seed
)

_bs = cfg["train"]["batch_size"]
if len(tr_idx) < _bs * 3:
    _bs = max(8, len(tr_idx) // 8)
    cfg["train"]["batch_size"] = _bs
    print(f"Adjusted batch_size to {_bs} (small train split)")

tr_loader, va_loader, te_loader = make_loaders(
    cohort,
    clin_x,
    tr_idx,
    va_idx,
    te_idx,
    batch_size=cfg["train"]["batch_size"],
)

device = cfg["train"].get("device") or auto_device()
model = MultimodalSCDModel(
    ModelConfig(
        clinical_input_dim=clin_x.shape[1],
        genomic_input_dim=cohort["genomic"].shape[1],
        imaging_input_dim=cohort["imaging"].shape[1],
        temporal_input_dim=cohort["temporal"].shape[2],
        embed_dim=cfg["model"]["embed_dim"],
        fusion=cfg["model"]["fusion"],
        dropout=cfg["model"]["dropout"],
        num_severity_classes=cfg["model"]["num_severity_classes"],
    )
)

trainer = Trainer(
    model,
    TrainConfig(
        epochs=cfg["train"]["epochs"],
        lr=cfg["train"]["lr"],
        weight_decay=cfg["train"]["weight_decay"],
        grad_clip=cfg["train"]["grad_clip"],
        alpha=cfg["train"]["alpha"],
        beta=cfg["train"]["beta"],
        early_stop_patience=cfg["train"]["early_stop_patience"],
        select_metric=cfg["train"]["select_metric"],
        device=device,
    ),
)

train_out = trainer.fit(tr_loader, va_loader)
print("Best val score:", train_out["best_score"])

ev = evaluate_model_full(model, te_loader, device=device)
metrics = ev["metrics"]
for k in sorted(metrics.keys()):
    v = metrics[k]
    if isinstance(v, (float, int, np.floating, np.integer)):
        print(f"{k}: {float(v):.4f}")

out_dir = Path(ROOT) / "experiments" / "results" / "colab_nigeria_ndhs2018"
out_dir.mkdir(parents=True, exist_ok=True)
torch.save(model.state_dict(), out_dir / "last_model.pt")
print("Saved checkpoint to", out_dir / "last_model.pt")



---

## Part 6 — Real cohort: Mali DHS 2018

Requires **MLHR7ADT.dta**. Imaging/temporal remain zero; severity from anemia classes.


## 2. Load **real** Mali EDS-VI / DHS 2018 rows (Household Recode)

Resolves ``MALI_DHS_MLHR_DTA`` from the environment, or **Colab file upload**. If neither is provided, raises with instructions (no synthetic fallback).

Optional: enforce ``PIP_INDEX_URL`` / offline installs if your runtime blocks PyPI — this section only uses ``pandas``.

In [ ]:
import os
from pathlib import Path

from mmvlm4scd.data import build_cohort_from_mali_dhs2018_hr


def resolve_mali_dhs_stata_path() -> Path:
    env = os.environ.get("MALI_DHS_MLHR_DTA")
    if env:
        p = Path(env).expanduser()
        if p.is_file():
            return p
        raise FileNotFoundError(f"MALI_DHS_MLHR_DTA points to missing path: {p}")
    cwd = Path("MLHR7ADT.dta").resolve()
    if cwd.is_file():
        return cwd
    if _in_colab():
        from google.colab import files as colab_files  # noqa: WPS433

        print("Upload MLHR7ADT.dta from MLHR7ADT.zip (DHS Mali 2018 Household Recode)…")
        up = colab_files.upload()
        names = list(up.keys())
        if len(names) != 1:
            raise RuntimeError("Please upload exactly one .dta file")
        wrote = Path("/content") / names[0]
        wrote.write_bytes(up[names[0]])
        return wrote
    raise FileNotFoundError(
        "Set MALI_DHS_MLHR_DTA to MLHR7ADT.dta or copy that file into the repo root."
    )


TIMESTEPS = 24

dta_path = resolve_mali_dhs_stata_path()
print("Stata:", dta_path)

cohort = build_cohort_from_mali_dhs2018_hr(
    dta_path,
    timesteps=TIMESTEPS,
    max_patients=None,
)

print(cohort["meta"])
print("Patients:", len(cohort["clinical"]))
cohort["clinical"].head()




## 3. Train and evaluate on real Mali DHS modalities

Mirrors ``run_full_experiment.py`` but loads **Mali DHS** children. Severity is **anemia class from Hb**, not sickle genotype (Mali 2018 lacks ``sb113b``). Imaging/temporal are zeros; **`beta=0`** disables Cox.


In [ ]:
import os
from pathlib import Path

import numpy as np
import torch

from mmvlm4scd.data import StandardPreprocessor
from mmvlm4scd.data.dataloaders import make_loaders
from mmvlm4scd.data.synthetic import split_indices
from mmvlm4scd.evaluation import evaluate_model_full
from mmvlm4scd.models import ModelConfig, MultimodalSCDModel
from mmvlm4scd.training import Trainer, TrainConfig
from mmvlm4scd.utils import auto_device, set_seed


_cfg_seed = int(os.environ.get("MMVLM_SEED", "7"))

cfg = {
    "model": {
        "embed_dim": 64,
        "fusion": "attention",
        "dropout": 0.1,
        "num_severity_classes": 3,
    },
    "train": {
        "epochs": 12,
        "batch_size": 64,
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "grad_clip": 1.0,
        "alpha": 1.0,
        # Mali DHS cohort: no real survival outcome here — Cox term disabled.
        "beta": 0.0,
        "early_stop_patience": 6,
        "select_metric": "auroc_ovr",
        "device": "cuda" if torch.cuda.is_available() else "cpu",
        "seed": _cfg_seed,
    },
}

set_seed(cfg["train"]["seed"])

pre = StandardPreprocessor().fit(cohort["clinical"])
clin_x = pre.transform(cohort["clinical"])
tr_idx, va_idx, te_idx = split_indices(
    len(cohort["severity"]), seed=_cfg_seed
)

_bs = cfg["train"]["batch_size"]
if len(tr_idx) < _bs * 3:
    _bs = max(8, len(tr_idx) // 8)
    cfg["train"]["batch_size"] = _bs
    print(f"Adjusted batch_size to {_bs} (small train split)")

tr_loader, va_loader, te_loader = make_loaders(
    cohort,
    clin_x,
    tr_idx,
    va_idx,
    te_idx,
    batch_size=cfg["train"]["batch_size"],
)

device = cfg["train"].get("device") or auto_device()
model = MultimodalSCDModel(
    ModelConfig(
        clinical_input_dim=clin_x.shape[1],
        genomic_input_dim=cohort["genomic"].shape[1],
        imaging_input_dim=cohort["imaging"].shape[1],
        temporal_input_dim=cohort["temporal"].shape[2],
        embed_dim=cfg["model"]["embed_dim"],
        fusion=cfg["model"]["fusion"],
        dropout=cfg["model"]["dropout"],
        num_severity_classes=cfg["model"]["num_severity_classes"],
    )
)

trainer = Trainer(
    model,
    TrainConfig(
        epochs=cfg["train"]["epochs"],
        lr=cfg["train"]["lr"],
        weight_decay=cfg["train"]["weight_decay"],
        grad_clip=cfg["train"]["grad_clip"],
        alpha=cfg["train"]["alpha"],
        beta=cfg["train"]["beta"],
        early_stop_patience=cfg["train"]["early_stop_patience"],
        select_metric=cfg["train"]["select_metric"],
        device=device,
    ),
)

train_out = trainer.fit(tr_loader, va_loader)
print("Best val score:", train_out["best_score"])

ev = evaluate_model_full(model, te_loader, device=device)
metrics = ev["metrics"]
for k in sorted(metrics.keys()):
    v = metrics[k]
    if isinstance(v, (float, int, np.floating, np.integer)):
        print(f"{k}: {float(v):.4f}")

out_dir = Path(ROOT) / "experiments" / "results" / "colab_mali_dhs2018"
out_dir.mkdir(parents=True, exist_ok=True)
torch.save(model.state_dict(), out_dir / "last_model.pt")
print("Saved checkpoint to", out_dir / "last_model.pt")

